# 02 — EDA and SQL Analytics

This notebook performs exploratory data analysis and SQL-based analytics on the cleaned startup funding dataset.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_raw_csv
from src.preprocessing import preprocess
from src.database import create_database, run_query
from src.config import CLEANED_CSV, DB_PATH

## 1. Load Cleaned Data

In [ ]:
if CLEANED_CSV.exists():
    df = pd.read_csv(CLEANED_CSV)
else:
    raw = load_raw_csv()
    df = preprocess(raw)

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Dataset Overview

In [ ]:
print("Dataset Info:")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Valid amounts: {df['has_valid_amount'].sum()}")
print(f"Total recorded funding: ${df['amount_clean'].sum() / 1e9:.2f}B")

## 3. Funding Events by Year

In [ ]:
yearly = df.groupby('year').agg(
    event_count=('sr_no', 'count'),
    total_funding=('amount_clean', 'sum'),
    median_funding=('amount_clean', 'median'),
    mean_funding=('amount_clean', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(yearly['year'], yearly['event_count'], color='steelblue')
axes[0].set_title('Funding Events by Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Event Count')

axes[1].bar(yearly['year'], yearly['total_funding'] / 1e9, color='coral')
axes[1].set_title('Total Recorded Funding by Year')

axes[1].set_ylabel('Total Funding (B USD)')

plt.tight_layout()
plt.savefig('../reports/funding_by_year.png', dpi=150)
plt.show()
print(yearly)

## 4. Funding Events by Industry

In [ ]:
industry = df.groupby('industry_clean').agg(
    event_count=('sr_no', 'count'),
    total_funding=('amount_clean', 'sum'),
    median_funding=('amount_clean', 'median')
).reset_index().sort_values('total_funding', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
top10 = industry.head(10)
ax.barh(top10['industry_clean'], top10['total_funding'] / 1e9, color='steelblue')
ax.set_title('Top 10 Industries by Total Recorded Funding')
ax.set_xlabel('Total Funding (B USD)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/funding_by_industry.png', dpi=150)
plt.show()
print(top10)

## 5. Funding Events by City

In [ ]:
city = df.groupby('city_clean').agg(
    event_count=('sr_no', 'count'),
    total_funding=('amount_clean', 'sum'),
    median_funding=('amount_clean', 'median')
).reset_index().sort_values('total_funding', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
top10 = city.head(10)
ax.barh(top10['city_clean'], top10['total_funding'] / 1e9, color='seagreen')
ax.set_title('Top 10 Cities by Total Recorded Funding')
ax.set_xlabel('Total Funding (B USD)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/funding_by_city.png', dpi=150)
plt.show()
print(top10)

## 6. Funding Type Distribution

In [ ]:
funding_type = df['investment_type_clean'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
funding_type.head(10).plot(kind='bar', ax=ax, color='purple')
ax.set_title('Top 10 Funding Types')
ax.set_xlabel('Funding Type')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/funding_type_distribution.png', dpi=150)
plt.show()
print(funding_type.head(10))

## 7. Top Investors

In [ ]:
investors = df['investors_name'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
investors.head(10).plot(kind='bar', ax=ax, color='orange')
ax.set_title('Top 10 Most Frequently Recorded Investors')
ax.set_xlabel('Investor')
ax.set_ylabel('Appearance Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/top_investors.png', dpi=150)
plt.show()
print(investors.head(10))
print("\nNote: Investor frequency does not imply quality or returns.")

## 8. Funding Amount Distribution

In [ ]:
valid_amounts = df['amount_clean'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(valid_amounts / 1e6, bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Funding Amount Distribution')
axes[0].set_xlabel('Amount (Million USD)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log10(valid_amounts), bins=50, color='coral', edgecolor='black')
axes[1].set_title('Log10 Funding Amount Distribution')
axes[1].set_xlabel('Log10(Amount USD)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../reports/amount_distribution.png', dpi=150)
plt.show()

print(f"Median: ${valid_amounts.median():,.0f}")
print(f"Mean: ${valid_amounts.mean():,.0f}")
print(f"Std: ${valid_amounts.std():,.0f}")
print(f"Skewness: {valid_amounts.skew():.2f}")

## 9. SQL Database Setup and Queries

In [ ]:
db_path = create_database(df)
print(f"Database created at: {db_path}")

In [ ]:
# Annual trends
query = "SELECT * FROM v_yearly_summary ORDER BY year"
result = run_query(db_path, query)
print(result)

In [ ]:
# Sector summary
query = "SELECT * FROM v_sector_summary ORDER BY total_funding DESC"
result = run_query(db_path, query)
print(result.head(10))

In [ ]:
# City summary
query = "SELECT * FROM v_city_summary ORDER BY total_funding DESC"
result = run_query(db_path, query)
print(result.head(10))

## 10. Sector-by-Year Comparison

In [ ]:
sector_year = df.groupby(['industry_clean', 'year']).agg(
    event_count=('sr_no', 'count'),
    total_funding=('amount_clean', 'sum')
).reset_index()

top_industries = df.groupby('industry_clean')['amount_clean'].sum().nlargest(5).index
pivot = sector_year[sector_year['industry_clean'].isin(top_industries)].pivot(
    index='year', columns='industry_clean', values='event_count'
)

fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind='bar', ax=ax)
ax.set_title('Top 5 Industries: Events by Year')
ax.set_xlabel('Year')
ax.set_ylabel('Event Count')
ax.legend(title='Industry')
plt.tight_layout()
plt.savefig('../reports/sector_by_year.png', dpi=150)
plt.show()

## 11. Key Findings

1. **E-Commerce and FinTech** dominate recorded funding totals
2. **Bengaluru** leads in both event count and total funding
3. **Seed and Series A** are the most common funding types
4. Funding amounts are highly right-skewed (long tail of large rounds)
5. **2017-2018** saw peak funding activity
6. Investor frequency reflects appearance count, not quality